## (In)homogeneous Poisson process experimental protocol

In [ ]:
# Imports
import numpy as np
import time
from scipy.stats import poisson
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output
from statsmodels.tsa.stattools import acf
from neuroplatform import StimShape, StimParam , IntanSofware, Trigger, Database, StimPolarity, Experiment

In [ ]:
token = '<token>'
exp = Experiment(token)
print(f'Eletrodes: {exp.electrodes}')

In [ ]:
# Attribute list for Stim obj
print(StimParam().__doc__)

In [ ]:
# Example electrode setup
first_stim_channel, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
first_stim_channel = StimParam()
first_stim_channel.index = exp.electrodes[1]
first_stim_channel.trigger_key = 0
first_stim_channel.phase_amplitude1 = 10
first_stim_channel.phase_amplitude2 = 10
first_stim_channel.display_attributes()

print(last_updated_paramaters)

In [ ]:
second_stim_channel, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
second_stim_channel = StimParam()
second_stim_channel.index = exp.electrodes[9]
second_stim_channel.trigger_key = 1
second_stim_channel.phase_amplitude1 = 10
second_stim_channel.phase_amplitude2 = 10
second_stim_channel.display_attributes()

print(last_updated_paramaters)

In [ ]:
# Exponential refractory period function
def exponential_refractory(time_since_last_spike, tau_ref=5):
    """
    Calculate exponential refractory period:

            Paramaters:
                :param time_since_last_spike: time since last update in ms
                :param tau_ref (Time Constant): modelled refractory period in ms
            Returns:
                :return: returns exponential decay variable based on - time / refractory
    """
    return np.exp(-time_since_last_spike / tau_ref)

In [ ]:
def sinusoidal_rate(time_step, period, baseline=22.5, amplitude=17.5):
    """
    Calculates a sinusoidal firing rate for an IPP, oscillating between 5 Hz and 40 Hz.

            Parameters:
                :param time_step: time step in ms
                :param period: period in ms
                :param baseline: baseline in ms
                :param amplitude: amplitude in ms
            Returns:
                :return: returns sinusoidal firing rate
    """
    return amplitude * np.sin(2 * np.pi * time_step / period) + baseline


In [ ]:
def generate_spike_train(rate, duration, bin_size=0.001, tau_ref=5, rate_function=None, period=None):
    """
    Generates a spike train, either HPP or IPP depending on rate_function.

            Parameters:
                :param rate: rate in Hz | Constant for HPP | Initial rate for IPP
                :param duration: duration in S
                :param bin_size: bin size in S
                :param tau_ref: time constant <- Used for refractory period
                :param rate_function: rate function | Optional
                :param period: oscillation period in ms for the rate function

            Returns:
                :return: spike train | 1D-Array of size duration, each element is bin. 1 = Pulse, 0, No-Pulse
    """
    num_bins = int(duration / bin_size)

    # Use rate_function or constant rate
    if rate_function is None:           # HPP
        spikes_per_bin = poisson.rvs(mu=rate * bin_size, size=num_bins)
    else:                               # IPP
        spikes_per_bin = np.zeros(num_bins)

        # Randomise the period if not provided
        if period is None:
            period = np.random.uniform(5000, 15000)  # Random period between 5000ms and 15000ms -adjust range as needed-

        # Simulates temporal passing for sin wave
        for i in range(num_bins):
            time_step = i * bin_size * 1000
            rate_at_time = sinusoidal_rate(time_step, period)
            spikes_per_bin[i] = poisson.rvs(mu=rate_at_time * bin_size)


    # Init spike train
    spike_train = np.zeros(num_bins, dtype=np.uint8)
    last_spike_bin = -1

    # Loop over 1D array, adding spikes where the poisson process has indicated
    for i, spike_count in enumerate(spikes_per_bin):
        if spike_count > 0:
            time_since_last_spike = (i - last_spike_bin) * bin_size * 1000
            refractory_factor = exponential_refractory(time_since_last_spike)
            prob_spike = 1 - refractory_factor
            spike_train[i] = np.random.choice([0, 1], p=[refractory_factor, prob_spike])  # Bernoulli trial
            if spike_train[i] == 1:
                last_spike_bin = i

    return spike_train


In [ ]:
# Variables

'''
Rate: Hz rate for HPP stim
Duration: duration in Seconds of stimulation
Bin_size: Resolution for which Poisson processes will be calculated
'''
rate = 3
duration = 600
bin_size = 0.001

# Pattern generation (Using the IPP pattern generator)
electrode1_pattern = generate_spike_train(rate, duration, rate_function=sinusoidal_rate, period=3000)
# Pattern generation (Using the HPP pattern generator)
electrode2_pattern = generate_spike_train(rate, duration)

# Sample train length
np.sum(electrode1_pattern)

In [ ]:
# Main Code block

intan = IntanSofware()
trigger_gen = Trigger()

stim_param_list = [first_stim_channel, second_stim_channel]
spike_trains = [electrode1_pattern, electrode2_pattern] # Add more as needed

intan.set_count([0])

try:
    if exp.start(): # Signal the start of an experiment to all users
        # Measure impedance
        intan.impedance()

        # Disable Variation STD (keep a fixed threshold)
        intan.var_threshold(False)

        # Send stim parameter
        intan.send_stimparam(stim_param_list)
        start_exp = datetime.utcnow()

        # Trigger 0 = electrode 0 by indexing
        trigger0 = np.zeros(16, dtype=np.uint8)
        trigger1 = np.zeros(16, dtype=np.uint8)
        #trigger2 = np.zeros(16, dtype=np.uint8)
        #trigger3 = np.zeros(16, dtype=np.uint8)
        #trigger4 = np.zeros(16, dtype=np.uint8)
        #trigger5 = np.zeros(16, dtype=np.uint8)
        #trigger6 = np.zeros(16, dtype=np.uint8)
        #trigger7 = np.zeros(16, dtype=np.uint8)
        trigger0[0] = 1
        trigger1[1] = 1
        #trigger2[2] = 1
        #trigger3[3] = 1
        #trigger4[0] = 1
        #trigger5[0] = 1
        #trigger6[0] = 1
        #trigger7[0] = 1

        # Main stimulation loop
        start_time = time.time()  # Get current time
        while time.time() - start_time < duration:
            current_bin = int((time.time() - start_time) / bin_size)

            # Iterate through spike trains and trigger if spike present
            if electrode1_pattern[current_bin] == 1:
                trigger_gen.send(trigger0)
            if electrode2_pattern[current_bin] == 1:
                trigger_gen.send(trigger1)
            '''if electrode3_pattern[current_bin] == 1:
                trigger_gen.send(trigger2)
            if electrode4_pattern[current_bin] == 1:
                trigger_gen.send(trigger3)
            if electrode5_pattern[current_bin] == 1:
                trigger_gen.send(trigger4)
            if electrode6_pattern[current_bin] == 1:
                trigger_gen.send(trigger5)
            if electrode7_pattern[current_bin] == 1:
                trigger_gen.send(trigger6)
            if electrode8_pattern[current_bin] == 1:
                trigger_gen.send(trigger7)'''

            time.sleep(bin_size)        # Wait for the duration of the bin

        count_spikes = intan.read_count()
        # count the number of spike of the NeuroSphere
        total_spikes = np.sum(count_spikes[exp.electrodes[0]:exp.electrodes[31]])
        clear_output(True)
        print(f'# Spikes: {total_spikes}')

        stop_exp = datetime.utcnow()

        # Disable all stims
        for stim in stim_param_list:
            stim.enable = False
        intan.send_stimparam(stim_param_list)

finally:
    trigger_gen.close()
    intan.var_threshold(True)
    intan.close()
    exp.stop()

print(f"Experiment start: " + start_exp)

print(f"Experiment end: " + stop_exp)